# Listing Dataiku projects and info for migration

This script is designed to be run on a Dataiku DSS Design Node v12 and later.

It will produce a table at the bottom that can be directly exported to CSV file by clicking the "Export this dataframe" button.

I was not able to test it in an active DSS v12.6 environment with connections to Hive, so I cannot guarantee that the column "NUMBER_OF_HIVE_RECIPES_USED_IN_THIS_PROJECT" is accurate. However, I did include columns that list all the unique types of recipes and datasets used in each project.

This script produces a table with the following columns:
* PROJECT_KEY - the unique identifier used to identify a Dataiku project.
* Project Name - the human friendly project name that is displayed in the projects list.
* Project URL - a placeholder URL where you should replace REPLACE_ME_WITH_FQDN with the proper hostname and/or port
* NUMBER_OF_HIVE_RECIPES_USED_IN_THIS_PROJECT
* Dataset_types_used_in_project
* Owner	- project owner's Dataiku username
* DATE_PROJECT_WAS_LAST_MODIFIED
* Recipe_types_used_in_project

# Instructions for use

1) Upload this .ipynb file to any existing project. It will not make any changes. Open it.

2) From the top menu in the Jupyter Notebook, select Kernel > Restart & Run All

3) Scroll to near the bottom. After the last code cell and before the table starts there should be an "Export this dataframe" button. Click it and download a CSV file.

In [0]:
from datetime import datetime
import dataiku
import pandas as pd

In [ ]:
# Connect to the DSS instance
client = dataiku.api_client()

In [ ]:
project_info = []

for project_key in client.list_project_keys():
    try:
        project = client.get_project(project_key)
        project_metadata = project.get_metadata()
        project_summary = project.get_summary()

        display_name = project_metadata.get('label', 'Unknown Project Name')

        last_modified_timestamp = project_summary.get('versionTag', {}).get('lastModifiedOn')
        if last_modified_timestamp:
            last_modified_date = datetime.fromtimestamp(last_modified_timestamp / 1000).strftime('%Y-%m-%d %H:%M:%S')
        else:
            last_modified_date = 'unknown_date'

        recipes = project.list_recipes()
        datasets = project.list_datasets()

        hive_count = sum(1 for r in recipes if r['type'].lower() == 'hive')
        dataset_types = {d.get('type') for d in datasets}
        recipe_types = {r.get('type') for r in recipes}

        project_info.append({
            'PROJECT_KEY': project_key,
            'Project Name': display_name,
            # Replace REPLACE_ME_WITH_FQDN with your DSS instance hostname.
            'Project URL': f"https://REPLACE_ME_WITH_FQDN/projects/{project_key}/",
            'NUMBER_OF_HIVE_RECIPES_USED_IN_THIS_PROJECT': hive_count,
            'Dataset_types_used_in_project': dataset_types,
            'Owner': project.get_permissions().get('owner', 'unknown owner'),
            'DATE_PROJECT_WAS_LAST_MODIFIED': last_modified_date,
            'Recipe_types_used_in_project': recipe_types,
        })
    except Exception as exc:
        print(f"Error processing project {project_key}: {exc}")

df = pd.DataFrame(project_info)
df

# Click "Export this dataframe" below to download as CSV.